In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

e:\AI-VENV\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
import os

os.environ["HF_HOME"] = r"E:\Qwen\huggingface"
os.environ["HF_HUB_CACHE"] = r"E:\Qwen\huggingface\hub"

In [4]:
MODEL_NAME = "Qwen/Qwen3-8B"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

In [15]:
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.float32,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True,
)

Loading weights: 100%|██████████| 399/399 [01:55<00:00,  3.44it/s]


In [8]:
dataset = load_dataset(
    "json",
    data_files="./dataset/V1A.jsonl",
    
)

Generating train split: 48114 examples [00:00, 92574.25 examples/s]


In [9]:
import json

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()

model.gradient_checkpointing_enable()

model.config.use_cache = False


SYSTEM_PROMPT = (
    "You are an expert memory extraction assistant.\n"
    "Extract only stable long-term memories from the conversation.\n"
    "Do NOT summarize temporary events.\n"
    "Return ONLY a JSON object in exactly this format:\n"
    "{\"memories\": [\"...\"]}\n"
    "If no long-term memory exists, return:\n"
    "{\"memories\": []}"
)


def formatting_func(example):

    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, indent=2)}"
    )

    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        indent=2
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_input,
        },
        {
            "role": "assistant",
            "content": assistant_output,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [11]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [12]:


new_dataset = load_dataset(
    "json",
    data_files="./dataset/V1A.jsonl",
  
)




new_lengths = analyze_dataset(
    "NEW V4A DATASET",
    new_dataset,
    tokenizer,
    formatting_func
)


NEW V4A DATASET
Processed 5000/48114
Processed 10000/48114
Processed 15000/48114
Processed 20000/48114
Processed 25000/48114
Processed 30000/48114
Processed 35000/48114
Processed 40000/48114
Processed 45000/48114

----- BASIC -----
Samples              : 48114
Average Tokens       : 429.96
Median Tokens        : 435.0
Minimum Tokens       : 140
Maximum Tokens       : 913

----- TOKEN DISTRIBUTION -----
>256 tokens          : 41462
>512 tokens          : 9912
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 100.29 sec
Average/sample       : 0.303 ms
Samples/sec          : 479.75

----- LONGEST SAMPLES -----
Sample   3845 : 913 tokens
Sample  11831 : 913 tokens
Sample  13697 : 913 tokens
Sample  17591 : 913 tokens
Sample  42411 : 913 tokens
Sample   7305 : 912 tokens
Sample  25923 : 912 tokens
Sample  32228 : 912 tokens
Sample  40197 : 912 tokens
Sample  18024 : 911 tokens
